## 01 — Data Ingestion
- Load documents from text, CSV, PDF, JSON, web, SQL, S3, and Git sources, then apply cleaning + metadata enrichment.

In [ ]:
import sys
import os
sys.path.append("..")

from rag_pipeline.ingestion import (
    load_documents, clean_and_filter_documents, enrich_metadata,
)
from rag_pipeline.utils import setup_logging
setup_logging()

In [ ]:
# Text
text_docs = load_documents([{
    "type": "text",
    "path": "/content/a-survey-of-retrieval-augmented-generation-RAG/data/drake_lyrics.txt",
}])
print(text_docs[0].page_content[:300])

In [ ]:
# CSV
csv_docs = load_documents([{
    "type": "csv",
    "path": "/content/a-survey-of-retrieval-augmented-generation-RAG/data/arxiv_data.csv",
    "content_columns": ["summaries"],  
}])[:500]
print(csv_docs[2].page_content[:200])
print(csv_docs[2].metadata)

In [ ]:
# PDF
pdf_docs = load_documents([{"type": "pdf", "path": "/content/a-survey-of-retrieval-augmented-generation-RAG/data/ENGINEERING/10030015.pdf"}])
print(len(pdf_docs), "pages")
print(pdf_docs[0].page_content[:300])

In [ ]:
# JSON
json_docs = load_documents([{
    "type": "json",
    "path": "/content/a-survey-of-retrieval-augmented-generation-RAG/data/drake_data.json",
    "jq_schema": ".[] | .lyrics",
    "text_content": False,
}])
print(len(json_docs), "songs")
print(json_docs[0].page_content[:200])

In [ ]:
# Web 
web_docs = load_documents([{
    "type": "web",
    "urls": [
        "https://en.wikipedia.org/wiki/Python_(programming_language)",
        "https://en.wikipedia.org/wiki/Java_(programming_language)",
    ],
    "parse_only_classes": ["firstHeading", "mw-parser-output"],
}])
print([d.metadata.get("source") for d in web_docs])

In [ ]:
# Cleaning + enrichment
cleaned  = clean_and_filter_documents(csv_docs, min_length=50, max_length=10000)
enriched = enrich_metadata(cleaned)

print("Documents after clean:", len(cleaned))
if enriched:                             
    print("Enriched sample metadata:", enriched[0].metadata)
else:
    print("No documents survived cleaning — check content_columns.")